In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
import pickle

from sklearn.metrics import r2_score
from lightgbm import LGBMRegressor
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import VotingRegressor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

class CONFIG:
    seed = 42
    target_col = "responder_6"
    feature_cols = ["symbol_id", "time_id"] \
        + [f"feature_{idx:02d}" for idx in range(79)] \
        + [f"responder_{idx}_lag_1" for idx in range(9)]
    categorical_cols = []

/home/jx1421/JS-Kaggle-JX/js-venv/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pl.scan_parquet("train.parquet").collect().to_pandas()
valid = pl.scan_parquet("valid.parquet").collect().to_pandas()
train.shape, valid.shape

((21022056, 104), (1082224, 104))

In [3]:
X_train = train[ CONFIG.feature_cols ]
y_train = train[ CONFIG.target_col ]
w_train = train[ "weight" ]
X_valid = valid[ CONFIG.feature_cols ]
y_valid = valid[ CONFIG.target_col ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

((21022056, 90),
 (21022056,),
 (21022056,),
 (1082224, 90),
 (1082224,),
 (1082224,))

In [4]:
def get_model(seed):
    # XGBoost parameters
    XGB_Params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'n_estimators': 200,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 1,
        'reg_lambda': 5,
        'random_state': seed,
        'tree_method': 'gpu_hist',
        'device' : 'cuda',
        'n_gpus' : 2,
    }
    
    XGB_Model = XGBRegressor(**XGB_Params)
    return XGB_Model

In [5]:
import random
N_MODELS = 5         # Number of models to train
FEATURE_FRAC = 0.7   # Fraction of features to select for each model
ROW_FRAC = 0.9       # Fraction of rows to sample (for bagging)

models = []
subsets_of_features = []

for i in range(N_MODELS):
    # ---------------------
    # 1) Random subset of features
    # ---------------------
    # Determine how many features to pick
    k_features = int(len(CONFIG.feature_cols) * FEATURE_FRAC)
    # Pick random features
    selected_features = random.sample(CONFIG.feature_cols, k=k_features)
    
    # ---------------------
    # 2) (Optional) Bagging - random subset of rows
    # ---------------------
    # Sample (with or without replacement) some fraction of rows
    # For example, random rows without replacement:
    n_rows = int(len(X_train) * ROW_FRAC)
    sampled_idx = np.random.choice(X_train.index, size=n_rows, replace=False)
    
    X_sub = X_train.loc[sampled_idx, selected_features]
    y_sub = y_train.loc[sampled_idx]
    w_sub = w_train.loc[sampled_idx]
    
    # ---------------------
    # Train an XGBoost model on this subset
    # ---------------------
    model = get_model(CONFIG.seed + i)
    model.fit(X_sub, y_sub, sample_weight=w_sub)
    
    # Keep track of trained model and subset of features
    models.append(model)
    subsets_of_features.append(selected_features)



Valid score

In [6]:
preds_list = []
for model, feats in zip(models, subsets_of_features):
    preds_list.append(model.predict(X_valid[feats]))

# Average predictions
ensemble_preds = np.mean(preds_list, axis=0)
r2_score(y_valid, ensemble_preds, sample_weight=w_valid)

0.006875589774509527

Train score

In [7]:
preds_list = []
for model, feats in zip(models, subsets_of_features):
    preds_list.append(model.predict(X_train[feats]))

# Average predictions
ensemble_preds = np.mean(preds_list, axis=0)
r2_score(y_train, ensemble_preds, sample_weight=w_train)

0.033178999955918886

Valid score

In [8]:
preds_list = []
for model, feats in zip(models, subsets_of_features):
    preds_list.append(model.predict(X_valid[feats]))

# Average predictions
ensemble_preds = np.mean(preds_list, axis=0)
r2_score(y_valid, ensemble_preds, sample_weight=w_valid)

0.006875589774509527

# Now change symbol_id to dummy categorical and refit naive XGb

In [9]:
train["symbol_id"] = train["symbol_id"].astype('category')
valid["symbol_id"] = valid["symbol_id"].astype('category')
X_train = train[ CONFIG.feature_cols ]
y_train = train[ CONFIG.target_col ]
w_train = train[ "weight" ]
X_valid = valid[ CONFIG.feature_cols ]
y_valid = valid[ CONFIG.target_col ]
w_valid = valid[ "weight" ]

X_train.shape, y_train.shape, w_train.shape, X_valid.shape, y_valid.shape, w_valid.shape

((21022056, 90),
 (21022056,),
 (21022056,),
 (1082224, 90),
 (1082224,),
 (1082224,))

# Enable XGb to deal with categorical

In [10]:
def get_model(seed):
    # XGBoost parameters
    XGB_Params = {
        'learning_rate': 0.05,
        'max_depth': 6,
        'n_estimators': 200,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 1,
        'reg_lambda': 5,
        'random_state': seed,
        'tree_method': 'gpu_hist',
        'device' : 'cuda',
        'n_gpus' : 2,
        'enable_categorical': True,
        'use_label_encoder': False
    }
    
    XGB_Model = XGBRegressor(**XGB_Params)
    return XGB_Model

In [11]:
N_MODELS = 5         # Number of models to train
FEATURE_FRAC = 0.7   # Fraction of features to select for each model
ROW_FRAC = 0.9      # Fraction of rows to sample (for bagging)

models = []
subsets_of_features = []

for i in range(N_MODELS):
    # Determine how many features to pick
    k_features = int(len(CONFIG.feature_cols) * FEATURE_FRAC)
    # Pick random features
    selected_features = random.sample(CONFIG.feature_cols, k=k_features)

    # bagging
    n_rows = int(len(X_train) * ROW_FRAC)
    sampled_idx = np.random.choice(X_train.index, size=n_rows, replace=False)
    
    X_sub = X_train.loc[sampled_idx, selected_features]
    y_sub = y_train.loc[sampled_idx]
    w_sub = w_train.loc[sampled_idx]
    
    # Train an XGBoost model on this subset
    model = get_model(CONFIG.seed + i)
    model.fit(X_sub, y_sub, sample_weight=w_sub)
    
    # Keep track of trained model and subset of features
    models.append(model)
    subsets_of_features.append(selected_features)



# Train score after categorical and random forest

In [12]:
preds_list = []
for model, feats in zip(models, subsets_of_features):
    preds_list.append(model.predict(X_train[feats]))

# Average predictions
ensemble_preds = np.mean(preds_list, axis=0)
r2_score(y_train, ensemble_preds, sample_weight=w_train)

0.034912504289385415

# Valid score

In [13]:
preds_list = []
for model, feats in zip(models, subsets_of_features):
    preds_list.append(model.predict(X_valid[feats]))

# Average predictions
ensemble_preds = np.mean(preds_list, axis=0)
r2_score(y_valid, ensemble_preds, sample_weight=w_valid)

0.006868563465593169

# Combine train and valid, and re-train random forest XGB on whole dataset

In [14]:
train = pd.concat([train, valid]).reset_index(drop=True)
X_train = train[ CONFIG.feature_cols ]
y_train = train[ CONFIG.target_col ]
w_train = train[ "weight" ]
print(X_train.dtypes)

symbol_id            category
time_id                 int16
feature_00            float32
feature_01            float32
feature_02            float32
                       ...   
responder_4_lag_1     float32
responder_5_lag_1     float32
responder_6_lag_1     float32
responder_7_lag_1     float32
responder_8_lag_1     float32
Length: 90, dtype: object


In [15]:
N_MODELS = 5         # Number of models to train
FEATURE_FRAC = 0.7   # Fraction of features to select for each model
ROW_FRAC = 0.9       # Fraction of rows to sample (for bagging)

models = []
subsets_of_features = []

for i in range(N_MODELS):
    # Determine how many features to pick
    k_features = int(len(CONFIG.feature_cols) * FEATURE_FRAC)
    # Pick random features
    selected_features = random.sample(CONFIG.feature_cols, k=k_features)

    # bagging
    n_rows = int(len(X_train) * ROW_FRAC)
    sampled_idx = np.random.choice(X_train.index, size=n_rows, replace=False)
    
    X_sub = X_train.loc[sampled_idx, selected_features]
    y_sub = y_train.loc[sampled_idx]
    w_sub = w_train.loc[sampled_idx]
    
    # Train an XGBoost model on this subset
    model = get_model(CONFIG.seed + i)
    model.fit(X_sub, y_sub, sample_weight=w_sub)
    
    # Keep track of trained model and subset of features
    models.append(model)
    subsets_of_features.append(selected_features)



# Train score on grand dataset

In [16]:
preds_list = []
for model, feats in zip(models, subsets_of_features):
    preds_list.append(model.predict(X_train[feats]))

# Average predictions
ensemble_preds = np.mean(preds_list, axis=0)
r2_score(y_train, ensemble_preds, sample_weight=w_train)

0.033290919062544666

# Save rand forest

In [17]:
import pickle

ensemble_result = {
    "models": models,  # e.g. a list of RandomForest estimators
    "subsets_of_features": subsets_of_features  # the feature subsets used for each model
}

with open("XGB_ensemble.pkl", "wb") as fp:
    pickle.dump(ensemble_result, fp)


In [18]:
model_path = "XGB_ensemble.pkl"
with open(model_path, "rb") as fp:
    saved_ensemble = pickle.load(fp)
    forests_loaded = saved_ensemble["models"]
    features_forest_loaded = saved_ensemble["subsets_of_features"]